In [25]:
import numpy as np
rng = np.random.default_rng(1)

In [34]:
for i in range(10):
    rng = np.random.default_rng(1)
    print(rng.beta(100,1000, 1))

[0.09265788]
[0.09265788]
[0.09265788]
[0.09265788]
[0.09265788]
[0.09265788]
[0.09265788]
[0.09265788]
[0.09265788]
[0.09265788]


In [33]:
rng = np.random.default_rng(1)
for i in range(10):
    print(rng.beta(100,1000, 1))

[0.09265788]
[0.09982088]
[0.09362194]
[0.08588134]
[0.09306424]
[0.08742993]
[0.07012865]
[0.09276038]
[0.077057]
[0.10089545]


# Bandit benchmark notebook

Этот ноутбук запускает benchmark-пайплайн (аналогично `src/run_benchmark.py`) на вашем датасете.

## 1) Настройки

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
"""Benchmark helpers for logged ad bandit data (polars processing + pandas metrics)."""

from __future__ import annotations

from dataclasses import dataclass
import math
import random
from typing import Callable, Literal

import pandas as pd
import polars as pl

try:
    from tqdm.auto import tqdm
except Exception:  # noqa: BLE001
    tqdm = None

Action = int
NULL_FEATURE_FILL = 0.0


@dataclass
class ScenarioConfig:
    name: str
    pretrain_source: Literal["random", "all", "none"]
    online_update: bool


class BasePolicy:
    can_update_online: bool = True

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del candidates, features, row
        raise NotImplementedError

    def select_batch(
        self,
        candidates_batch: list[list[Action]],
        features_batch: list[list[float]],
        rows_batch: list[dict[str, object]],
    ) -> list[Action]:
        if not (len(candidates_batch) == len(features_batch) == len(rows_batch)):
            raise ValueError("Batch inputs must have equal length")
        return [
            int(self.select(candidates, features, row))
            for candidates, features, row in zip(candidates_batch, features_batch, rows_batch)
        ]

    def update(self, action: Action, reward: float, features: list[float] | None = None) -> None:
        del action, reward, features

    def update_batch(self, pending_updates: list[tuple[int, float, list[float]]]) -> None:
        for a, r, f in pending_updates:
            self.update(a, r, f)

    def fit(self, train_df: pl.DataFrame) -> None:
        pending_updates = [
            (int(r["show"]), float(r["reward"]), r["features_list"])
            for r in train_df.iter_rows(named=True)
        ]
        self.update_batch(pending_updates)


class EpsilonGreedyPolicy(BasePolicy):
    def __init__(self, epsilon: float = 0.1, seed: int = 42):
        self.epsilon = epsilon
        self.rng = random.Random(seed)
        self.counts: dict[Action, int] = {}
        self.values: dict[Action, float] = {}

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del features, row
        if not candidates:
            raise ValueError("Empty candidate set")
        if self.rng.random() < self.epsilon:
            return self.rng.choice(candidates)
        return max(candidates, key=lambda a: self.values.get(a, 0.0))

    def update(self, action: Action, reward: float, features: list[float] | None = None) -> None:
        del features
        n = self.counts.get(action, 0) + 1
        v = self.values.get(action, 0.0)
        self.values[action] = v + (reward - v) / n
        self.counts[action] = n


class UCBPolicy(BasePolicy):
    def __init__(self, exploration: float = 2.0):
        self.exploration = exploration
        self.t = 0
        self.counts: dict[Action, int] = {}
        self.values: dict[Action, float] = {}

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del features, row
        if not candidates:
            raise ValueError("Empty candidate set")
        for a in candidates:
            if self.counts.get(a, 0) == 0:
                return a
        log_t = math.log(max(self.t, 1))
        return max(candidates, key=lambda a: self.values[a] + math.sqrt(self.exploration * log_t / self.counts[a]))

    def update(self, action: Action, reward: float, features: list[float] | None = None) -> None:
        del features
        self.t += 1
        n = self.counts.get(action, 0) + 1
        v = self.values.get(action, 0.0)
        self.values[action] = v + (reward - v) / n
        self.counts[action] = n


class ThompsonSamplingPolicy(BasePolicy):
    def __init__(self, alpha: float = 1.0, beta: float = 1.0, seed: int = 42):
        self.alpha0 = alpha
        self.beta0 = beta
        self.rng = random.Random(seed)
        self.alpha: dict[Action, float] = {}
        self.beta: dict[Action, float] = {}

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del features, row
        if not candidates:
            raise ValueError("Empty candidate set")
        return max(candidates, key=lambda a: self.rng.betavariate(self.alpha.get(a, self.alpha0), self.beta.get(a, self.beta0)))

    def update(self, action: Action, reward: float, features: list[float] | None = None) -> None:
        del features
        self.alpha[action] = self.alpha.get(action, self.alpha0) + reward
        self.beta[action] = self.beta.get(action, self.beta0) + (1.0 - reward)


class ContextualBanditPlaceholder(BasePolicy):
    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del candidates, features, row
        raise NotImplementedError("Contextual bandits are intentionally not implemented yet")




class LogisticTSPolicy(BasePolicy):
    """Wrapper over contextualbandits.online.LogisticTS.

    Train-once in this benchmark and no online updates.
    """

    can_update_online = False

    def __init__(self, random_seed: int = 42):
        self.random_seed = random_seed
        self._model = None
        self._fitted = False
        self._actions: list[int] = []
        self._a2i: dict[int, int] = {}

        self.a: list[int] = []
        self.r: list[int] = []
        self.f: list[list[float]] = []

    def update_batch(self, pending_updates: list[tuple[int, float, list[float]]]) -> None:
        import numpy as np
        try:
            from contextualbandits.online import LogisticTS
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError("contextualbandits is required for LogisticTSPolicy") from exc

        new_actions = {int(a) for a, _, _ in pending_updates}
        if not new_actions:
            raise ValueError("pending_updates contains no actions")

        for a in new_actions:
            if a not in self._a2i:
                self._a2i[a] = len(self._actions)
                self._actions.append(a)

        for a, r, f in pending_updates:
            self.a.append(self._a2i[a])
            self.r.append(int(float(r) > 0.0))
            self.f.append(f)

        self._model = LogisticTS(
            nchoices=len(self._actions),
            random_state=self.random_seed,
        )
        self._model.fit(np.array(self.f)[:, :50], np.array(self.a), np.array(self.r))

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        import numpy as np

        del row
        if not candidates:
            raise ValueError("Empty candidate set")
        if self._model is None:
            return int(np.random.choice(candidates))

        ids = [self._a2i[candidate] for candidate in candidates if self._a2i.get(candidate) is not None]
        if len(ids) == 0:
            return int(np.random.choice(candidates))
        probs = self._model.predict(np.array(features[:50]), output_all_scores=True)
        idx_max = probs["scores"][0][ids].argmax()
        best_action = self._actions[ids[idx_max]]

        return int(best_action)


class PartitionedTSPolicy(BasePolicy):
    """Wrapper over contextualbandits.online.PartitionedTS.

    Train-once in this benchmark and no online updates.
    """

    can_update_online = False

    def __init__(self, random_seed: int = 42):
        self.random_seed = random_seed
        self._model = None
        self._fitted = False
        self._actions: list[int] = []
        self._a2i: dict[int, int] = {}

    def fit(self, train_df: pl.DataFrame) -> None:
        if self._fitted:
            raise RuntimeError("PartitionedTSPolicy can only be trained once")
        if train_df.height == 0:
            self._fitted = True
            return

        try:
            from contextualbandits.online import PartitionedTS
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError("contextualbandits is required for PartitionedTSPolicy") from exc

        actions = sorted({int(r["show"]) for r in train_df.iter_rows(named=True)})
        if not actions:
            self._fitted = True
            return
        self._actions = actions
        self._a2i = {a: i for i, a in enumerate(actions)}

        X: list[list[float]] = []
        a: list[int] = []
        r: list[int] = []
        for row in train_df.iter_rows(named=True):
            action = int(row["show"])
            if action not in self._a2i:
                continue
            X.append(list(row["features_list"]))
            a.append(self._a2i[action])
            r.append(int(float(row["reward"]) > 0.0))

        if not X:
            self._fitted = True
            return

        model = PartitionedTS(nchoices=len(self._actions), random_state=self.random_seed)
        model.fit(X, a, r)
        self._model = model
        self._fitted = True

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del row
        if not candidates:
            raise ValueError("Empty candidate set")
        if self._model is None:
            return candidates[0]

        probs = self._model.predict_proba([list(features)])[0]
        best_action = candidates[0]
        best_score = -1e18
        for a in candidates:
            idx = self._a2i.get(int(a))
            score = float(probs[idx]) if idx is not None else 0.0
            if score > best_score:
                best_score = score
                best_action = int(a)
        return best_action


class CatBoostPolicy(BasePolicy):
    """Gradient boosting policy based on CatBoostClassifier.

    Model is train-once: repeated fit calls are forbidden.
    """

    can_update_online = False

    def __init__(self, random_seed: int = 42):
        self.random_seed = random_seed
        self._model = None
        self._fitted = False

    @staticmethod
    def _row_to_vector(features: list[float], action: int) -> list[float]:
        return list(features) + [float(action)]

    def fit(self, train_df: pl.DataFrame) -> None:
        if self._fitted:
            raise RuntimeError("CatBoostPolicy can only be trained once")
        if train_df.height == 0:
            self._fitted = True
            return

        try:
            from catboost import CatBoostClassifier
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError("catboost is required for CatBoostPolicy") from exc

        X: list[list[float]] = []
        y: list[int] = []
        for row in train_df.iter_rows(named=True):
            action = int(row["show"])
            features = row["features_list"]
            X.append(self._row_to_vector(features, action))
            y.append(int(float(row["reward"]) > 0.0))

        if not X:
            self._fitted = True
            return

        model = CatBoostClassifier(
            iterations=200,
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            verbose=False,
            random_seed=self.random_seed,
        )
        model.fit(X, y)
        self._model = model
        self._fitted = True

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del row
        if not candidates:
            raise ValueError("Empty candidate set")
        if self._model is None:
            return candidates[0]

        best_action = candidates[0]
        best_score = -1.0
        for a in candidates:
            vec = self._row_to_vector(features, int(a))
            p = float(self._model.predict_proba([vec])[0][1])
            if p > best_score:
                best_score = p
                best_action = int(a)
        return best_action

    def update(self, action: Action, reward: float, features: list[float] | None = None) -> None:
        del action, reward, features
        return


def _parse_candidates(raw: str) -> list[int]:
    if raw is None or raw == "":
        return []
    return [int(x) for x in str(raw).split("\\t") if str(x) != ""]


def _parse_features(raw: str) -> list[float]:
    if raw is None or raw == "":
        return []

    vals: list[float] = []
    for x in str(raw).split("\\t"):
        token = str(x).strip().lower()
        if token == "":
            continue
        if token in {"null", "none", "nan"}:
            vals.append(NULL_FEATURE_FILL)
        else:
            vals.append(float(token))
    return vals


def preprocess_bandit_dataframe(df: pl.DataFrame) -> pl.DataFrame:
    req = {"policy", "reward", "features", "show", "candidates", "date"}
    missing = req - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    prepared = df.with_columns(
        [
            pl.col("show").cast(pl.Int64),
            pl.col("reward"),
            pl.col("date").str.to_datetime(strict=False),
            pl.col("candidates").map_elements(_parse_candidates, return_dtype=pl.List(pl.Int64)).alias("candidates_list"),
            pl.col("features").map_elements(_parse_features, return_dtype=pl.List(pl.Float64)).alias("features_list"),
        ]
    )

    return prepared.with_columns(
        (pl.lit(1.0) / pl.col("candidates_list").list.len().cast(pl.Float64)).alias("propensity")
    )




def apply_standard_scaler_to_features(df: pl.DataFrame) -> pl.DataFrame:
    """Scale `features_list` with StandardScaler when sklearn is available."""
    try:
        import numpy as np
        from sklearn.preprocessing import StandardScaler

        feat_rows = df.select("features_list").to_series().to_list()
        non_empty_idx = [i for i, row in enumerate(feat_rows) if isinstance(row, list) and len(row) > 0]
        if not non_empty_idx:
            return df

        dim = len(feat_rows[non_empty_idx[0]])
        valid_idx = [i for i in non_empty_idx if len(feat_rows[i]) == dim]
        if not valid_idx:
            return df

        X = np.array([feat_rows[i] for i in valid_idx], dtype=float)
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
        for j, i in enumerate(valid_idx):
            feat_rows[i] = [float(v) for v in Xs[j].tolist()]
        return df.with_columns(pl.Series("features_list", feat_rows))
    except Exception:
        return df
def split_train_test_by_date(df: pl.DataFrame, test_ratio: float = 0.2) -> tuple[pl.DataFrame, pl.DataFrame]:
    if not 0.0 < test_ratio < 1.0:
        raise ValueError("test_ratio must be in (0,1)")
    ordered = df.sort("date")
    split_idx = int(ordered.height * (1.0 - test_ratio))
    return ordered.slice(0, split_idx), ordered.slice(split_idx, ordered.height - split_idx)


def select_pretrain_data(train_df: pl.DataFrame, source: Literal["random", "all", "none"]) -> pl.DataFrame:
    if source == "none":
        return train_df.clear()
    if source == "all":
        return train_df
    if source == "random":
        return train_df.filter(pl.col("policy") == "random")
    raise ValueError(f"Unknown pretrain source: {source}")


def build_expected_reward_estimator(train_df: pl.DataFrame) -> Callable[[dict[str, object], Action], float]:
    sums: dict[int, float] = {}
    counts: dict[int, int] = {}
    for row in train_df.iter_rows(named=True):
        a = int(row["show"])
        r = float(row["reward"])
        sums[a] = sums.get(a, 0.0) + r
        counts[a] = counts.get(a, 0) + 1

    total_sum = sum(sums.values())
    total_n = sum(counts.values())
    global_mean = (total_sum / total_n) if total_n > 0 else 0.0

    def estimate(_row: dict[str, object], action: Action) -> float:
        n = counts.get(action, 0)
        return (sums[action] / n) if n > 0 else global_mean

    return estimate




def build_random_action_ctr_stats(df: pl.DataFrame) -> tuple[dict[int, float], float]:
    random_df = df.filter(pl.col("policy") == "random") if "policy" in df.columns else df
    sums: dict[int, float] = {}
    counts: dict[int, int] = {}
    for row in random_df.iter_rows(named=True):
        a = int(row["show"])
        r = float(row["reward"])
        sums[a] = sums.get(a, 0.0) + r
        counts[a] = counts.get(a, 0) + 1

    ctr_by_action: dict[int, float] = {}
    for a, n in counts.items():
        ctr_by_action[a] = sums.get(a, 0.0) / n if n > 0 else 0.0
    max_ctr = max(ctr_by_action.values()) if ctr_by_action else 0.0
    return ctr_by_action, max_ctr
def evaluate_policy(
    policy: BasePolicy,
    test_df: pl.DataFrame,
    online_update: bool,
    env_reward: Callable[[dict[str, object], Action], float] | None = None,
    show_progress: bool = True,
    progress_desc: str = "evaluate",
    ctr_by_action: dict[int, float] | None = None,
    max_random_ctr: float = 0.0,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    total_reward = 0.0
    ips_weighted_reward_sum = 0.0
    used = 0
    replay_matches = 0
    cumulative_regret = 0.0  # replay regret accumulator (used only for regret metrics)
    cumulative_ips_regret = 0.0  # IPS regret accumulator (used only for regret metrics)
    history_rows: list[dict[str, float | int]] = []

    action_ctr = ctr_by_action or {}
    # Regret-only baseline fallback for unseen actions within candidate sets.

    total_steps = max(test_df.height, 1)
    update_chunk = max(1, int(total_steps * 0.10))
    progress_chunk = max(1, int(total_steps * 0.05))

    pbar = None
    if show_progress and tqdm is not None:
        pbar = tqdm(total=test_df.height, desc=progress_desc, leave=False, dynamic_ncols=True, mininterval=0.5)

    pending_updates: list[tuple[int, float, list[float]]] = []
    next_progress_mark = progress_chunk

    test_rows = list(test_df.iter_rows(named=True))

    batch_size = max(1, min(512, update_chunk))

    for batch_start in range(0, len(test_rows), batch_size):
        rows_batch = test_rows[batch_start : batch_start + batch_size]
        candidates_batch = [row["candidates_list"] for row in rows_batch]
        features_batch = [row["features_list"] for row in rows_batch]
        actions_batch = policy.select_batch(candidates_batch, features_batch, rows_batch)
        if len(actions_batch) != len(rows_batch):
            raise ValueError("select_batch must return one action per input row")

        for offset, row in enumerate(rows_batch):
            step = batch_start + offset + 1
            candidates = candidates_batch[offset]
            features = features_batch[offset]
            action = int(actions_batch[offset])

            logged_reward = float(row["reward"])
            logged_match = int(action == int(row["show"]))
            propensity = float(row.get("propensity", 0.0) or 0.0)

            ips_reward = (logged_match * logged_reward / propensity) if propensity > 0 else 0.0
            ips_weighted_reward_sum += ips_reward
            # Regret-only per-step baseline: max expected CTR among currently available actions.
            candidate_ctrs = [action_ctr.get(int(a), max_random_ctr) for a in candidates] if candidates else [max_random_ctr]
            step_max_ctr = max(candidate_ctrs) if candidate_ctrs else max_random_ctr
            ips_step_regret = step_max_ctr - (logged_reward if logged_match else 0.0)
            cumulative_ips_regret += ips_step_regret

            if env_reward is None:
                if not logged_match:
                    if pbar is not None and step >= next_progress_mark:
                        pbar.update(step - pbar.n)
                        next_progress_mark += progress_chunk
                    continue
                reward = logged_reward
                replay_matches += 1
                regret = step_max_ctr - reward
            else:
                reward = float(env_reward(row, action))
                replay_matches += int(action == int(row["show"]))
                regret = step_max_ctr - reward

            total_reward += reward
            cumulative_regret += regret
            used += 1

            if online_update and policy.can_update_online:
                pending_updates.append((action, reward, features))
                if len(pending_updates) >= update_chunk:
                    policy.update_batch(pending_updates)
                    pending_updates.clear()

            history_rows.append(
                {
                    "step": step,
                    "reward": reward,
                    "avg_reward": total_reward / used,
                    "ips_reward": ips_reward,
                    "ips_avg_reward": ips_weighted_reward_sum / step,
                    "cumulative_regret": cumulative_regret,
                    "avg_regret": cumulative_regret / used,
                    "cumulative_ips_regret": cumulative_ips_regret,
                    "avg_ips_regret": cumulative_ips_regret / step,
                }
            )

            if pbar is not None and step >= next_progress_mark:
                pbar.update(step - pbar.n)
                next_progress_mark += progress_chunk

    if pending_updates:
        policy.update_batch(pending_updates)

    if pbar is not None:
        pbar.update(test_df.height - pbar.n)
        pbar.close()

    ctr = total_reward / used if used else 0.0
    ips_ctr = ips_weighted_reward_sum / test_df.height if test_df.height else 0.0
    match_rate = replay_matches / test_df.height if test_df.height else 0.0
    final_avg_regret = (cumulative_regret / used) if used else 0.0
    final_avg_ips_regret = (cumulative_ips_regret / test_df.height) if test_df.height else 0.0
    metrics_df = pd.DataFrame([
        {
            "impressions_total": test_df.height,
            "impressions_used": used,
            "total_reward": total_reward,
            "ctr": ctr,
            "ips_weighted_reward": ips_weighted_reward_sum,
            "ips_ctr": ips_ctr,
            "replay_match_rate": match_rate,
            "cumulative_regret": cumulative_regret,
            "avg_regret": final_avg_regret,
            "cumulative_ips_regret": cumulative_ips_regret,
            "avg_ips_regret": final_avg_ips_regret,
        }
    ])
    history_df = pd.DataFrame(history_rows)
    return metrics_df, history_df


def run_scenarios(
    train_df: pl.DataFrame,
    test_df: pl.DataFrame,
    policy_factories: dict[str, Callable[[], BasePolicy]],
    scenarios: list[ScenarioConfig],
    env_reward: Callable[[dict[str, object], Action], float] | None = None,
    show_progress: bool = True,
) -> dict[str, pd.DataFrame]:
    metrics_parts: list[pd.DataFrame] = []
    history_parts: list[pd.DataFrame] = []

    for scenario in scenarios:
        pretrain_df = select_pretrain_data(train_df, scenario.pretrain_source)
        # Regret-only statistics are estimated from combined train+test logs.
        ctr_source = pl.concat([train_df.select(["policy", "show", "reward"]), test_df.select(["policy", "show", "reward"])], how="vertical")
        ctr_by_action, max_random_ctr = build_random_action_ctr_stats(ctr_source)

        for algo_name, make_policy in policy_factories.items():
            policy = make_policy()
            if pretrain_df.height > 0:
                policy.fit(pretrain_df)

            metrics_df, history_df = evaluate_policy(
                policy=policy,
                test_df=test_df,
                online_update=scenario.online_update,
                env_reward=env_reward,
                show_progress=show_progress,
                progress_desc=f"{scenario.name}/{algo_name}",
                ctr_by_action=ctr_by_action,
                max_random_ctr=max_random_ctr,
            )
            metrics_df["scenario"] = scenario.name
            metrics_df["algo"] = algo_name
            metrics_parts.append(metrics_df)

            if not history_df.empty:
                history_df["scenario"] = scenario.name
                history_df["algo"] = algo_name
                history_parts.append(history_df)

    out_metrics = pd.concat(metrics_parts, ignore_index=True) if metrics_parts else pd.DataFrame()
    out_history = pd.concat(history_parts, ignore_index=True) if history_parts else pd.DataFrame()
    return {"metrics": out_metrics, "history": out_history}


def make_simulated_environment(
    proba_predictor: Callable[[dict[str, object], Action], float],
    stochastic: bool = True,
    seed: int = 42,
) -> Callable[[dict[str, object], Action], float]:
    rng = random.Random(seed)

    def env_reward(row: dict[str, object], action: Action) -> float:
        p = max(0.0, min(1.0, float(proba_predictor(row, action))))
        if not stochastic:
            return p
        return 1.0 if rng.random() < p else 0.0

    return env_reward


def default_five_scenarios() -> list[ScenarioConfig]:
    return [
        ScenarioConfig("case_1_random_pretrain_predict_only", "random", False),
        ScenarioConfig("case_2_random_pretrain_online_update", "random", True),
        ScenarioConfig("case_3_all_pretrain_predict_only", "all", False),
        ScenarioConfig("case_4_all_pretrain_online_update", "all", True),
        ScenarioConfig("case_5_no_pretrain_online_update", "none", True),
    ]


def core_scenarios() -> list[ScenarioConfig]:
    return [ScenarioConfig("case_2_random_pretrain_online_update", "random", True)]


In [3]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import polars as pl

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from run_benchmark import load_dataset

In [9]:
# Укажите путь к данным (.tsv или .parquet)
DATA_PATH = ROOT / 'data' / 'events.tsv'
PREPARED_TRAIN_PATH = ROOT / 'artifacts' / 'datasets' / 'train_prepared.parquet'
PREPARED_TEST_PATH = ROOT / 'artifacts' / 'datasets' / 'test_prepared.parquet'
USE_PREPARED_SPLITS = True
TEST_RATIO = 0.5
EPSILON = 0.1
SEED = 42
SIMULATE = False
STOCHASTIC_SIM = True
FULL_SCENARIOS = False

## 2) Загрузка и препроцессинг

In [5]:
if USE_PREPARED_SPLITS and PREPARED_TRAIN_PATH.exists() and PREPARED_TEST_PATH.exists():
    train_df = pl.read_parquet(PREPARED_TRAIN_PATH)
    test_df = pl.read_parquet(PREPARED_TEST_PATH)
    print('loaded prepared splits')
else:
    # Stage 1: load -> split -> filter -> save
    train_s1, test_s1 = stage1_make_splits(str(DATA_PATH), str(ROOT / 'artifacts' / 'datasets'), TEST_RATIO, SEED)
    # Stage 2: scale features and save prepared datasets
    train_final, test_final = stage2_scale_features(train_s1, test_s1, str(ROOT / 'artifacts' / 'datasets'))
    train_df = pl.read_parquet(train_final)
    test_df = pl.read_parquet(test_final)

print('train:', train_df.height, 'test(random only):', test_df.height)


loaded prepared splits
train: 12701839 test(random only): 3925311


## 3) Конфиг политик и сценариев

In [30]:
policy_factories = {
    # 'epsilon_greedy': lambda: EpsilonGreedyPolicy(epsilon=EPSILON, seed=SEED),
    # 'ucb': lambda: UCBPolicy(),
    # 'thompson_sampling': lambda: ThompsonSamplingPolicy(seed=SEED),
}

# try:
#     import catboost  # noqa: F401
#     policy_factories['catboost'] = lambda: CatBoostPolicy(random_seed=SEED)
# except Exception:
#     print('catboost is unavailable: skipping CatBoostPolicy')

try:
    import contextualbandits  # noqa: F401
    # policy_factories['logistic_ts'] = lambda: LogisticTSPolicy(random_seed=SEED)
    policy_factories['partitioned_ts'] = lambda: PartitionedTSPolicy(random_seed=SEED)
except Exception:
    print('contextualbandits is unavailable: skipping LogisticTS/PartitionedTS')


scenarios = default_five_scenarios() if FULL_SCENARIOS else core_scenarios()

## 4) Запуск benchmark

In [7]:
class LogisticTSPolicy(BasePolicy):
    """Wrapper over contextualbandits.online.LogisticTS.

    Train-once in this benchmark and no online updates.
    """

    can_update_online = False

    def __init__(self, random_seed: int = 42):
        self.random_seed = random_seed
        self._model = None
        self._fitted = False
        self._actions: list[int] = []
        self._a2i: dict[int, int] = {}

        self.a: list[int] = []
        self.r: list[int] = []
        self.f: list[list[float]] = []

    def update_batch(self, pending_updates: list[tuple[int, float, list[float]]]) -> None:
        import numpy as np
        try:
            from contextualbandits.online import LogisticTS
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError("contextualbandits is required for LogisticTSPolicy") from exc

        new_actions = {int(a) for a, _, _ in pending_updates}
        if not new_actions:
            raise ValueError("pending_updates contains no actions")

        for a in new_actions:
            if a not in self._a2i:
                self._a2i[a] = len(self._actions)
                self._actions.append(a)

        for a, r, f in pending_updates:
            self.a.append(self._a2i[a])
            self.r.append(int(float(r) > 0.0))
            self.f.append(f)

        self._model = LogisticTS(
            nchoices=len(self._actions),
            random_state=self.random_seed,
        )
        self._model.fit(np.array(self.f)[:, :50], np.array(self.a), np.array(self.r))

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        import numpy as np

        del row
        if not candidates:
            raise ValueError("Empty candidate set")
        if self._model is None:
            return int(np.random.choice(candidates))

        ids = [self._a2i[candidate] for candidate in candidates if self._a2i.get(candidate) is not None]
        if len(ids) == 0:
            return int(np.random.choice(candidates))
        probs = self._model.predict(np.array(features[:50]), output_all_scores=True)
        idx_max = probs["scores"][0][ids].argmax()
        best_action = self._actions[ids[idx_max]]

        return int(best_action)

    def select_batch(
        self,
        candidates_batch: list[list[Action]],
        features_batch: list[list[float]],
        rows_batch: list[dict[str, object]],
    ) -> list[Action]:
        import numpy as np
    
        if not (len(candidates_batch) == len(features_batch) == len(rows_batch)):
            raise ValueError("Batch inputs must have equal length")
    
        n = len(candidates_batch)
        if n == 0:
            return []
    
        del rows_batch
    
        if self._model is None:
            return [int(np.random.choice(cands)) if cands else None for cands in candidates_batch]  # type: ignore[misc]
            
        X = np.asarray([f[:50] for f in features_batch], dtype=np.float64)  # shape (n, d)
        probs = self._model.predict(X, output_all_scores=True)
        scores = probs["scores"] 
        
        out: list[Action] = []
        for i, candidates in enumerate(candidates_batch):
            if not candidates:
                raise ValueError("Empty candidate set in batch")
            ids = [self._a2i[candidate] for candidate in candidates if self._a2i.get(candidate) is not None]
            if not ids:
                out.append(int(np.random.choice(candidates)))
                continue
    
            row_scores = scores[i]
            best_local = int(np.argmax(row_scores[ids]))
            best_action = self._actions[ids[best_local]]
            out.append(int(best_action))
    
        return out
    
    # def select_batch(
    #     self,
    #     candidates_batch: list[list[Action]],
    #     features_batch: list[list[float]],
    #     rows_batch: list[dict[str, object]],
    # ) -> list[Action]:
    #     if not (len(candidates_batch) == len(features_batch) == len(rows_batch)):
    #         raise ValueError("Batch inputs must have equal length")

    #     if self._model is None:
    #         return int(np.random.choice(candidates))

    #     ids = [self._a2i[candidate] for candidate in candidates if self._a2i.get(candidate) is not None]
    #     if len(ids) == 0:
    #         return int(np.random.choice(candidates))
    #     probs = self._model.predict(np.array(features[:50]), output_all_scores=True)
    #     idx_max = probs["scores"][0][ids].argmax()
    #     best_action = self._actions[ids[idx_max]]

    #     return int(best_action)
    #     return [
    #         int(self.select(candidates, features, row))
    #         for candidates, features, row in zip(candidates_batch, features_batch, rows_batch)
    #     ]

In [32]:
class PartitionedTSPolicy(BasePolicy):
    """Wrapper over contextualbandits.online.PartitionedTS.

    Train-once in this benchmark and no online updates.
    """

    can_update_online = True

    def __init__(self, random_seed: int = 42):
        self.random_seed = random_seed
        self._model = None
        self._fitted = False
        self._actions: list[int] = []
        self._a2i: dict[int, int] = {}

        self.a: list[int] = []
        self.r: list[int] = []
        self.f: list[list[float]] = []

    def update_batch(self, pending_updates: list[tuple[int, float, list[float]]]) -> None:
        import numpy as np
        try:
            from contextualbandits.online import LogisticTS
        except Exception as exc:  # noqa: BLE001
            raise RuntimeError("contextualbandits is required for LogisticTSPolicy") from exc

        new_actions = {int(a) for a, _, _ in pending_updates}
        if not new_actions:
            raise ValueError("pending_updates contains no actions")

        for a in new_actions:
            if a not in self._a2i:
                self._a2i[a] = len(self._actions)
                self._actions.append(a)

        for a, r, f in pending_updates:
            self.a.append(self._a2i[a])
            self.r.append(int(float(r) > 0.0))
            self.f.append(f)
        from contextualbandits.online import PartitionedTS
        self._model = PartitionedTS(nchoices=len(self._actions), random_state=self.random_seed)
        self._model.fit(np.array(self.f)[:, :50], np.array(self.a), np.array(self.r))

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        import numpy as np

        del row
        if not candidates:
            raise ValueError("Empty candidate set")
        if self._model is None:
            return int(np.random.choice(candidates))

        ids = [self._a2i[candidate] for candidate in candidates if self._a2i.get(candidate) is not None]
        if len(ids) == 0:
            return int(np.random.choice(candidates))
        probs = self._model.predict(np.array(features[:50]), output_all_scores=True)
        idx_max = probs["scores"][0][ids].argmax()
        best_action = self._actions[ids[idx_max]]

        return int(best_action)

    def select_batch(
        self,
        candidates_batch: list[list[Action]],
        features_batch: list[list[float]],
        rows_batch: list[dict[str, object]],
    ) -> list[Action]:
        import numpy as np

        if not (len(candidates_batch) == len(features_batch) == len(rows_batch)):
            raise ValueError("Batch inputs must have equal length")

        n = len(candidates_batch)
        if n == 0:
            return []

        del rows_batch

        if self._model is None:
            out_random: list[Action] = []
            for cands in candidates_batch:
                if not cands:
                    raise ValueError("Empty candidate set in batch")
                out_random.append(int(np.random.choice(cands)))
            return out_random

        X = np.asarray([f[:50] for f in features_batch], dtype=np.float64)
        probs = self._model.predict(X, output_all_scores=True)
        scores = probs["scores"]

        out: list[Action] = []
        for i, candidates in enumerate(candidates_batch):
            if not candidates:
                raise ValueError("Empty candidate set in batch")
            ids = [self._a2i[candidate] for candidate in candidates if self._a2i.get(candidate) is not None]
            if not ids:
                out.append(int(np.random.choice(candidates)))
                continue

            row_scores = scores[i]
            best_local = int(np.argmax(row_scores[ids]))
            best_action = self._actions[ids[best_local]]
            out.append(int(best_action))

        return out

In [41]:

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
import numpy as np
import polars as pl

from scipy.optimize import minimize
from scipy.special import expit  # sigmoid


class OnlineLogisticRegression:
    """
    Online Logistic Regression with diagonal precision q (Laplace-like).
    y is expected in {-1, +1}

    Posterior approx:
      w ~ N(m, alpha^2 * diag(q)^-1)
    """

    def __init__(self, lambda_: float, alpha: float, n_dim: int, seed: int | None = None):
        self.lambda_ = float(lambda_)
        self.alpha = float(alpha)
        self.n_dim = int(n_dim)

        self.m = np.zeros(self.n_dim, dtype=np.float64)
        self.q = np.ones(self.n_dim, dtype=np.float64) * self.lambda_

        self.rng = np.random.default_rng(seed)
        self.w = self.get_weights()  # init sample

    def loss(self, w: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
        prior = 0.5 * (self.q * (w - self.m)).dot(w - self.m)
        z = y * (X @ w)
        ll = np.logaddexp(0.0, -z).sum()
        return float(prior + ll)

    def grad(self, w: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        g = self.q * (w - self.m)
        z = y * (X @ w)
        coeff = -y / (1.0 + np.exp(z))  # shape (n,)
        g += (coeff[:, None] * X).sum(axis=0)
        return g

    def get_weights(self) -> np.ndarray:
        # std = alpha / sqrt(q)
        std = self.alpha / np.sqrt(np.maximum(self.q, 1e-12))
        return self.rng.normal(loc=self.m, scale=std, size=self.n_dim)

    def fit(self, X: np.ndarray, y: np.ndarray, maxiter: int = 20) -> None:
        """
        One "online" update using batch X,y.
        """
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.int64)
        if X.ndim != 2 or X.shape[1] != self.n_dim:
            raise ValueError(f"X must be (n,{self.n_dim}), got {X.shape}")
        if y.ndim != 1 or y.shape[0] != X.shape[0]:
            raise ValueError("y must be (n,) aligned with X")

        # Step 1: find MAP w, start from previous w
        res = minimize(
            fun=self.loss,
            x0=self.w,
            args=(X, y),
            jac=self.grad,
            method="L-BFGS-B",
            options={"maxiter": int(maxiter), "disp": False},
        )
        self.w = res.x.astype(np.float64, copy=False)
        self.m = self.w.copy()

        # Step 2: update q (diagonal precision)
        # FIX vs your code: P should be sigmoid(X @ m), not sigmoid(X @ m - 1)
        P = expit(X @ self.m)           # shape (n,)
        v = P * (1.0 - P)               # shape (n,)
        self.q = self.q + (v[:, None] * (X * X)).sum(axis=0)

    def predict_proba(self, X: np.ndarray, mode: str = "sample") -> np.ndarray:
        X = np.asarray(X, dtype=np.float64)
        if X.ndim == 1:
            X = X.reshape(1, -1)

        if mode == "sample":
            w = self.get_weights()
        elif mode == "expected":
            w = self.m
        else:
            raise ValueError("mode not recognized: use 'sample' or 'expected'")

        p = expit(X @ w)
        return np.vstack([1.0 - p, p]).T


class LaplaceThompsonViaBayesianLogRegPolicy(BasePolicy):
    """
    Политика LogisticTS, которая ДОобучается всегда:
      - update(): сразу model.fit на одной строке
      - update_batch(): сразу model.fit на батче (группируем по action)
      - select(): TS — сэмплим веса через model.predict_proba(mode="sample")
    """

    can_update_online: bool = True

    def __init__(
        self,
        lambda_: float = 1.0,
        alpha: float = 1.0,
        maxiter_update: int = 5,        # итерации L-BFGS на 1 update (чтобы не было адски медленно)
        maxiter_batch: int = 20,        # итерации на батч-обучение
        seed: int | None = None,
    ) -> None:
        self.lambda_ = float(lambda_)
        self.alpha = float(alpha)
        self.maxiter_update = int(maxiter_update)
        self.maxiter_batch = int(maxiter_batch)
        self.seed = seed

        self._d: int | None = None
        self._models: Dict[int, OnlineLogisticRegression] = {}

    def _get_model(self, a: int) -> OnlineLogisticRegression:
        m = self._models.get(a)
        if m is None:
            arm_seed = None if self.seed is None else (self.seed + 1000003 * a)
            m = OnlineLogisticRegression(self.lambda_, self.alpha, self._d, seed=arm_seed)
            self._models[a] = m
        return m

    def _ensure_dim(self, features: list[float]) -> int:
        d = len(features)
        if self._d is None:
            self._d = d
        elif self._d != d:
            raise ValueError(f"Feature dimension changed: expected {self._d}, got {d}")
        return d

    def update(self, action: Action, reward: float, features: list[float] | None = None) -> None:
        if features is None:
            return
        self._ensure_dim(features)
        a = int(action)
        model = self._get_model(a)

        x = np.asarray(features, dtype=np.float64).reshape(1, -1)
        y = np.asarray([1 if float(reward) > 0 else -1], dtype=np.int64)

        model.fit(x, y, maxiter=self.maxiter_update)

    def update_batch(self, pending_updates: list[tuple[int, float, list[float]]]) -> None:
        if not pending_updates:
            return
            
        self._ensure_dim(pending_updates[0][2])
        by_arm: Dict[int, Tuple[List[np.ndarray], List[int]]] = {}
        for a, r, f in pending_updates:
            a = int(a)
            x = np.asarray(f, dtype=np.float64)
            y = 1 if float(r) > 0 else -1
            X_list, y_list = by_arm.setdefault(a, ([], []))
            X_list.append(x)
            y_list.append(y)

        for a, (X_list, y_list) in by_arm.items():
            model = self._get_model(a)
            X = np.vstack(X_list)
            y = np.asarray(y_list, dtype=np.int64)
            model.fit(X, y, maxiter=self.maxiter_batch)

    def select(self, candidates: list[Action], features: list[float], row: dict[str, object]) -> Action:
        del row
        if not candidates:
            raise ValueError("candidates is empty")

        x = np.asarray(features, dtype=np.float64).reshape(1, -1)

        best_a = int(candidates[0])
        best_score = -np.inf

        for a_raw in candidates:
            a = int(a_raw)
            model = self._get_model(a)
            p = float(model.predict_proba(x, mode="sample")[0, 1])  # TS score
            if p > best_score:
                best_score = p
                best_a = a

        return best_a

    # def fit(self, train_df: pl.DataFrame) -> None:
    #     # быстрая версия fit: берём списками и учим батчем
    #     if train_df.height == 0:
    #         return
    #     shows = train_df["show"].to_list()
    #     rewards = train_df["reward"].to_list()
    #     feats = train_df["features_list"].to_list()
    #     self.update_batch([(int(a), float(r), f) for a, r, f in zip(shows, rewards, feats)])

In [26]:
row = test_df[0]
candidates, reward, features = row["candidates_list"], row["reward"], row["features"]

In [17]:
train_df2 = train_df.sample(fraction=0.05, shuffle=True).sort("date")
test_df2 = test_df.sample(fraction=0.05, shuffle=True).sort("date")

In [36]:
policy_factories.clear()

In [37]:
policy_factories

{}

In [42]:
import numpy as np
# policy_factories['logistic_ts'] = lambda: LogisticTSPolicy(random_seed=SEED)
policy_factories['test'] = lambda: LaplaceThompsonViaBayesianLogRegPolicy(seed=SEED,
                                                                          lambda_ = 1.0, alpha = 1.0,
                                                                          maxiter_update = 5,
                                                                          maxiter_batch = 20)

env_reward = None
if SIMULATE:
    expected_fn = build_expected_reward_estimator(train_df)
    env_reward = make_simulated_environment(
        proba_predictor=expected_fn,
        stochastic=STOCHASTIC_SIM,
        seed=SEED,
    )

result = run_scenarios(
    train_df=train_df2,
    test_df=test_df2,
    policy_factories=policy_factories,
    scenarios=scenarios,
    env_reward=env_reward,
    show_progress=True,
)

metrics_df = result['metrics']
history_df = result['history']

display(metrics_df)
print('history rows:', len(history_df))

case_2_random_pretrain_online_update/

/var/folders/2h/f3088zln2y36htvcq0wjvt94_1rc11/T/ipykernel_1616/672068456.py:41: RuntimeWarning: overflow encountered in exp
  coeff = -y / (1.0 + np.exp(z))  # shape (n,)


,impressions_total,impressions_used,total_reward,ctr,ips_weighted_reward,ips_ctr,replay_match_rate,cumulative_regret,avg_regret,cumulative_ips_regret,avg_ips_regret,scenario,algo
0,196265,138812,2599.0,0.018723,4110.0,0.020941,0.707268,339.895022,0.002449,2352.373078,0.011986,case_2_random_pretrain_online_update,test


history rows: 138812


In [28]:
model = result._model

In [38]:
candidates

candidates_list
list[i64]
[8761]


In [46]:
for step, row in enumerate(test_df.iter_rows(named=True), start=1):
    candidates = row["candidates_list"]
    features = row["features_list"]
    break

In [47]:
ids = [result._a2i[candidate] for candidate in candidates if result._a2i.get(candidate) is not None]
ids, candidates

([23], [8761])

In [50]:
idx_max = res["scores"][0][ids].argmax()

In [52]:
result._actions[ids[idx_max]]

8761

In [29]:
res = result._model.predict(np.array(features), output_all_scores=True)
res

{'choice': array([42]),
 'score': array([[0.204529]]),
 'scores': array([[2.49512434e-03, 2.09844516e-02, 2.57080654e-02, 4.55609196e-02,
         2.83696953e-02, 1.21607761e-02, 7.64810874e-03, 1.59546464e-01,
         1.12195433e-02, 4.40301291e-02, 4.91423887e-02, 4.73602917e-02,
         8.76403378e-02, 1.59331373e-02, 1.64898253e-02, 1.71584038e-02,
         1.48368114e-02, 5.53483911e-02, 1.05575664e-02, 6.38352664e-03,
         7.61603148e-03, 1.41150423e-02, 1.43121902e-02, 1.36450953e-05,
         6.76098400e-03, 1.28806978e-01, 2.24669377e-02, 2.28732510e-02,
         8.40633337e-02, 1.21882892e-01, 3.07497066e-02, 1.97229108e-02,
         2.17156307e-02, 3.30909154e-03, 1.47114109e-02, 1.74628940e-02,
         1.89165172e-02, 4.39515878e-02, 3.80518007e-02, 1.39390342e-01,
         3.91360300e-02, 4.16792500e-04, 2.04528996e-01, 2.95429535e-02,
         1.66893860e-02, 7.18489599e-02, 3.83708716e-02, 3.60691436e-02,
         4.84183017e-02, 1.43864641e-01]])}

In [23]:
result.select(candidates, features, row)

ValueError: 'X' must be a numpy array or sparse CSR matrix.

## 5) Графики по сценариям

In [ ]:
if not history_df.empty:
    for scenario_name, part in history_df.groupby('scenario'):
        fig, axes = plt.subplots(2, 2, figsize=(14, 8))

        for algo, algo_df in part.groupby('algo'):
            max_step = int(algo_df['step'].max()) if len(algo_df) else 0
            stride = max(1, int(round(max_step * 0.05)))  # каждые 5%
            ds = algo_df.iloc[stride::stride] if len(algo_df) > stride else algo_df  # стартуем с 5%

            axes[0, 0].plot(ds['step'], ds['avg_reward'], label=algo)
            axes[0, 1].plot(ds['step'], ds['avg_regret'], label=algo)
            axes[1, 0].plot(ds['step'], ds['ips_avg_reward'], label=algo)
            axes[1, 1].plot(ds['step'], ds['avg_ips_regret'], label=algo)

        axes[0, 0].set_title(f'{scenario_name}: average reward')
        axes[0, 0].set_xlabel('step')
        axes[0, 0].set_ylabel('avg_reward')

        axes[0, 1].set_title(f'{scenario_name}: average regret')
        axes[0, 1].set_xlabel('step')
        axes[0, 1].set_ylabel('avg_regret')
        axes[0, 1].set_yscale('log')

        axes[1, 0].set_title(f'{scenario_name}: IPS average reward')
        axes[1, 0].set_xlabel('step')
        axes[1, 0].set_ylabel('ips_avg_reward')

        axes[1, 1].set_title(f'{scenario_name}: IPS average regret')
        axes[1, 1].set_xlabel('step')
        axes[1, 1].set_ylabel('avg_ips_regret')
        axes[1, 1].set_yscale('log')

        for ax in axes.ravel():
            ax.grid(True, alpha=0.3)
            ax.legend()

        fig.tight_layout()
        plt.show()
else:
    print('History is empty.')

## 6) Сохранение артефактов

In [ ]:
OUT_DIR = ROOT / 'artifacts'
OUT_DIR.mkdir(parents=True, exist_ok=True)

metrics_path = OUT_DIR / 'metrics.csv'
history_path = OUT_DIR / 'history.csv'

metrics_df.to_csv(metrics_path, index=False)
history_df.to_csv(history_path, index=False)

print('saved metrics:', metrics_path)
print('saved history:', history_path)